# Run the Models on the Validation Set

## Natural-language request

> Now run the models on the validation set.

This notebook takes the four models fitted on train in notebook 11 (logistic regression, random forest, XGBoost, CatBoost) and the flat-rule reference. It runs them **unchanged** on the validation set (the contract's **choose** period, arrivals 1 July to 31 December 2016), and does what the contract says val is for:

1. **Measure the probabilities on bookings the models never saw**, beside their training-set scores.
2. **Turn each method's probabilities into an oversell count per hotel-night** using the same rule for every method, and **choose each method's buffer on val** under the walk ceiling (project rules 5 and 6).
3. **Compare every method with the flat rule** on the same hotel-nights, in euros, empty rooms and walked guests, for the group and for each hotel.

## What stays fixed, and what is not claimed

- **Nothing is refitted.** The preprocessor and the models are loaded from notebooks 10 and 11, and the notebook checks that predicting val does not change them.
- **Test is never opened.** Its results come once, after every choice is frozen (project rule 2).
- **This notebook does not freeze a method.** It sets out the comparison; choosing and freezing the method is the team's decision, recorded in `docs/DECISIONS.md`.
- **The val euros are not a savings claim** (project rule 8). Each buffer is chosen on val and then scored on val, so the val figures flatter every method a little. The fair measure of any saving is the single test run.
- **Four inputs are provisional** (`is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled`, `company`), with timing not verified (notebook 06). Every result carries that caveat.
- **Full-book caveat** (project rule 7): the contract scores each night's final book, including bookings made after the forecast morning. This is a historical comparison, not a live morning forecast.

## Setup

Loads val from notebook 08, the fitted preprocessor from notebook 10 and the fitted models from notebook 11. `load_split` accepts only `"train"` and `"val"`, so test cannot be opened. Train is loaded only for the flat rule's hotel rates and to show training scores beside val scores.

In [1]:
from pathlib import Path
import json
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

# Find the repository so this works from the repo folder or notebooks/.
repo_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "src" / "contract.py").exists():
        repo_root = folder
        break
if repo_root is None:
    raise FileNotFoundError("Could not find the project src/ folder.")
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import contract

split_dir = repo_root / "data" / "splits"
prep_dir = repo_root / "data" / "preprocessed"
model_dir = repo_root / "data" / "models"
if not (model_dir / "train_scores.csv").exists():
    raise FileNotFoundError("data/models/ is missing. Run notebooks/11_train_models.ipynb first.")

manifest = json.loads((split_dir / "split_columns.json").read_text(encoding="utf-8"))
prep_info = json.loads((prep_dir / "feature_names.json").read_text(encoding="utf-8"))
inputs, label = manifest["inputs"], manifest["label"]


def load_split(name):
    # Test stays closed until every choice is frozen (project rule 2).
    if name not in ("train", "val"):
        raise ValueError("This notebook only opens train and val.")
    return pd.read_csv(split_dir / f"{name}.csv", index_col="row",
                       parse_dates=["arrival_date"], dtype={"agent": str, "company": str})


train = load_split("train")
val = load_split("val")

preprocessor = joblib.load(prep_dir / "preprocessor.joblib")
model_names = ["logistic regression", "random forest", "XGBoost", "CatBoost"]
models = {name: joblib.load(model_dir / f"{name.replace(' ', '_')}.joblib") for name in model_names}
train_scores = pd.read_csv(model_dir / "train_scores.csv", index_col="model")

# Values never seen in train go to the "infrequent" column (notebook 10, Step 9).
warnings.filterwarnings("ignore", message="Found unknown categories")
print(f"val: {len(val):,} bookings, {val.groupby(['hotel', 'arrival_date']).ngroups} hotel-nights, "
      f"arrivals {val['arrival_date'].min():%Y-%m-%d} to {val['arrival_date'].max():%Y-%m-%d}")

val: 29,546 bookings, 368 hotel-nights, arrivals 2016-07-01 to 2016-12-31


## Step 1: predict val with the fitted preprocessor and models

The models see **only the eight inputs** of each val booking. The label is not passed to the preprocessor or to any model. The flat-rule reference gives each booking its hotel's **train** cancellation rate.

**Leakage guards:**

1. Every val arrival lies in the choose period.
2. The preprocessor and every model are identical before and after predicting val, so nothing was refitted on val.
3. The prepared val table equals the one notebook 10 saved.
4. The prepared columns are exactly notebook 10's.

In [2]:
assert val["arrival_date"].min() >= pd.Timestamp(contract.TRAIN_END)
assert val["arrival_date"].max() < pd.Timestamp(contract.PROOF_START)

fingerprints = {"preprocessor": joblib.hash(preprocessor), **{n: joblib.hash(m) for n, m in models.items()}}

X_val_inputs = val[inputs]  # the eight inputs only; no label, date, price or identifier
assert label not in X_val_inputs.columns
X_val = preprocessor.transform(X_val_inputs)
assert list(X_val.columns) == prep_info["output_columns"]
saved = pd.read_csv(prep_dir / "val_X.csv", index_col="row")
assert saved.index.equals(X_val.index) and np.allclose(saved.to_numpy(dtype=float), X_val.to_numpy(dtype=float))

hotel_rate = train.groupby("hotel")[label].mean()
probabilities = {"flat-rule reference": val["hotel"].map(hotel_rate).to_numpy()}
for name, model in models.items():
    probabilities[name] = model.predict_proba(X_val.to_numpy(dtype=float))[:, 1]

assert fingerprints == {"preprocessor": joblib.hash(preprocessor), **{n: joblib.hash(m) for n, m in models.items()}}
for name, p in probabilities.items():
    assert len(p) == len(val) and np.all((p >= 0) & (p <= 1))

print("All four guards pass. Flat-rule rates from train:", {h: f"{r:.1%}" for h, r in hotel_rate.items()})
y_val = val[label]

All four guards pass. Flat-rule rates from train: {'City Hotel': '41.7%', 'Resort Hotel': '25.2%'}


## Step 2: probability scores on val, beside train

The same measures as notebook 11, now on bookings the models never saw. The **change from train** shows how much of each model's training score carried over. A model whose val score falls far below its train score was partly memorising train.

In [3]:
val_scores = pd.DataFrame({
    name: {
        "log loss": log_loss(y_val, p),
        "Brier score": brier_score_loss(y_val, p),
        "ROC AUC": roc_auc_score(y_val, p),
        "mean predicted rate": p.mean(),
        "actual rate": y_val.mean(),
    }
    for name, p in probabilities.items()
}).T

comparison = pd.DataFrame({
    "log loss (train)": train_scores["log loss"], "log loss (val)": val_scores["log loss"],
    "ROC AUC (train)": train_scores["ROC AUC"], "ROC AUC (val)": val_scores["ROC AUC"],
    "Brier (val)": val_scores["Brier score"],
    "mean predicted rate (val)": val_scores["mean predicted rate"], "actual rate (val)": val_scores["actual rate"],
}).loc[list(probabilities)]
display(comparison.style.format({c: "{:.4f}" for c in ["log loss (train)", "log loss (val)", "Brier (val)"]}
                                | {c: "{:.3f}" for c in ["ROC AUC (train)", "ROC AUC (val)"]}
                                | {c: "{:.1%}" for c in ["mean predicted rate (val)", "actual rate (val)"]}))

,log loss (train),log loss (val),ROC AUC (train),ROC AUC (val),Brier (val),mean predicted rate (val),actual rate (val)
flat-rule reference,0.6381,0.6520,0.582,0.557,0.2299,36.4%,36.9%
logistic regression,0.4622,0.6113,0.817,0.705,0.2108,28.1%,36.9%
random forest,0.4802,0.6012,0.810,0.701,0.2082,29.3%,36.9%
XGBoost,0.4576,0.6059,0.815,0.714,0.2112,27.8%,36.9%
CatBoost,0.4589,0.6056,0.815,0.702,0.2110,27.9%,36.9%


## Step 3: calibration on val

The same fixed probability bands as notebook 11: in each band, the mean predicted probability against the share that actually cancelled, with the number of val bookings. The last row is the average gap, weighted by bookings. Because the contract adds probabilities up, a gap here turns straight into too many or too few expected cancellations.

In [4]:
bands = np.linspace(0, 1, 11)
band_names = [f"{int(lo * 100)}-{int(hi * 100)}%" for lo, hi in zip(bands[:-1], bands[1:])]
calibration = {}
for name, p in probabilities.items():
    band = pd.cut(p, bands, labels=band_names, include_lowest=True)
    by_band = (pd.DataFrame({"predicted": p, "actual": y_val.to_numpy(), "band": band})
               .groupby("band", observed=False).agg(predicted=("predicted", "mean"), actual=("actual", "mean"),
                                                    bookings=("actual", "size")))
    cells = [f"{r.predicted:.0%} vs {r.actual:.0%} ({r.bookings:,})" if r.bookings else "-" for r in by_band.itertuples()]
    filled = by_band[by_band["bookings"] > 0]
    gap = ((filled["predicted"] - filled["actual"]).abs() * filled["bookings"]).sum() / filled["bookings"].sum()
    calibration[name] = cells + [f"{gap:.1%}"]
display(pd.DataFrame(calibration, index=[f"{b}: predicted vs actual (bookings)" for b in band_names] + ["average gap (weighted)"]))

,flat-rule reference,logistic regression,random forest,XGBoost,CatBoost
0-10%: predicted vs actual (bookings),-,"6% vs 13% (4,013)","8% vs 6% (1,194)","6% vs 6% (1,908)","6% vs 4% (1,667)"
10-20%: predicted vs actual (bookings),-,"14% vs 19% (5,428)","16% vs 16% (6,368)","15% vs 19% (7,235)","14% vs 18% (7,138)"
20-30%: predicted vs actual (bookings),"25% vs 28% (9,347)","28% vs 38% (5,529)","27% vs 35% (6,602)","27% vs 37% (5,069)","27% vs 38% (5,413)"
30-40%: predicted vs actual (bookings),-,"32% vs 41% (10,357)","32% vs 40% (11,354)","32% vs 41% (11,406)","32% vs 41% (11,382)"
40-50%: predicted vs actual (bookings),"42% vs 41% (20,199)","46% vs 73% (2,030)","45% vs 61% (1,847)","45% vs 69% (2,780)","45% vs 69% (2,771)"
50-60%: predicted vs actual (bookings),-,52% vs 48% (966),"53% vs 84% (1,778)",53% vs 81% (630),55% vs 76% (756)
60-70%: predicted vs actual (bookings),-,65% vs 55% (330),61% vs 65% (85),62% vs 52% (181),65% vs 52% (65)
70-80%: predicted vs actual (bookings),-,73% vs 86% (474),73% vs 100% (21),-,73% vs 14% (21)
80-90%: predicted vs actual (bookings),-,82% vs 73% (62),88% vs 100% (232),85% vs 0% (1),87% vs 86% (7)
90-100%: predicted vs actual (bookings),-,97% vs 94% (357),92% vs 100% (65),98% vs 99% (336),98% vs 99% (326)


## Step 4: expected vs actual cancellations per hotel-night

`contract.nightly_table` adds each val night's probabilities into the expected number of cancellations. This is the number the oversell count is built on. **Mean error** above zero means the method expects too many cancellations (pushing towards walks); below zero, too few (pushing towards empty rooms). Group and each hotel are shown separately (project rule 6).

In [5]:
scoring_columns = val[contract.REQUIRED]
nightly = {name: contract.nightly_table(scoring_columns, p) for name, p in probabilities.items()}

rows = []
for name, table in nightly.items():
    error = table["expected"] - table["cancelled"]
    for hotel, part in [("group", error), *error.groupby(level="hotel")]:
        rows.append({"hotel": hotel, "method": name, "hotel-nights": len(part),
                     "mean absolute error (rooms)": part.abs().mean(), "mean error (rooms)": part.mean(),
                     "nights within 1 room": (part.abs() <= 1).mean()})
nightly_scores = pd.DataFrame(rows).set_index(["hotel", "method"])
display(nightly_scores.style.format({"mean absolute error (rooms)": "{:.2f}", "mean error (rooms)": "{:+.2f}",
                                     "nights within 1 room": "{:.1%}"}))

,,hotel-nights,mean absolute error (rooms),mean error (rooms),nights within 1 room
hotel,method,,,,
group,flat-rule reference,368,9.19,-0.35,9.0%
City Hotel,flat-rule reference,184,13.27,+0.98,5.4%
Resort Hotel,flat-rule reference,184,5.11,-1.67,12.5%
group,logistic regression,368,9.83,-7.05,10.6%
City Hotel,logistic regression,184,14.09,-11.10,6.5%
Resort Hotel,logistic regression,184,5.58,-3.00,14.7%
group,random forest,368,9.27,-6.08,13.6%
City Hotel,random forest,184,13.17,-9.83,12.5%
Resort Hotel,random forest,184,5.37,-2.34,14.7%


## Step 5: the oversell rule, and choosing each buffer on val

**One rule for every method**, following the client's flat rule in `docs/EVALUATION_CONTRACT.md`:

> oversell count *k* = max(0, ⌊expected cancellations⌋ − *b*)

where *b* is the **buffer**: a fixed number of rooms held back for safety, the same on every night and at both hotels. The expected count is rounded **down** (⌊ ⌋) before the buffer is taken off, the cautious direction. The same rounding is used for every method, including the flat rule.

**Choosing *b* on val** (project rules 5 and 6), the same way for every method:

1. try every whole buffer from 0 to 60 rooms;
2. keep only the buffers that meet the **group-wide walk ceiling** of 10 walks per 1,000 arrivals on val;
3. of those, take the one with the **lowest total cost** in euros.

A method with no buffer that meets the ceiling is rejected. Only the group-wide ceiling is enforced: the client has given no per-hotel thresholds, so hotel results are reported, not judged (project rule 6).

In [6]:
BUFFERS = range(0, 61)


def oversell(table, buffer):
    return np.maximum(np.floor(table["expected"].to_numpy()) - buffer, 0)


sweep = []
for name, table in nightly.items():
    for buffer in BUFFERS:
        result = contract.night_cost(table, oversell(table, buffer))
        sweep.append({"method": name, "buffer": buffer, **result})
sweep = pd.DataFrame(sweep)

chosen = {}
for name, results in sweep.groupby("method", sort=False):
    allowed = results[results["ceiling_met"]]
    if allowed.empty:
        print(f"{name}: no buffer from 0 to 60 meets the ceiling; rejected.")
        continue
    best = allowed.loc[allowed["cost_eur"].idxmin()]
    chosen[name] = int(best["buffer"])
    if chosen[name] in (min(BUFFERS), max(BUFFERS)):
        print(f"{name}: chosen buffer {chosen[name]} is at the edge of the range tried; widen the range.")

display(pd.DataFrame({
    name: {"chosen buffer (rooms)": b,
           "smallest buffer meeting the ceiling": int(sweep[(sweep["method"] == name) & sweep["ceiling_met"]]["buffer"].min())}
    for name, b in chosen.items()
}).T)

,chosen buffer (rooms),smallest buffer meeting the ceiling
flat-rule reference,16,16
logistic regression,6,6
random forest,6,6
XGBoost,5,5
CatBoost,5,5


## Step 6: results on val at each method's chosen buffer

Euros, empty rooms, walked guests and walks per 1,000 arrivals, side by side (project rule 5), for the **group** (where the ceiling applies) and for **each hotel** at the same buffer. The last columns compare each method with the flat rule **on the same hotel-nights**.

These are **choose-period figures used for selection**, not savings: each buffer was chosen on these same nights.

In [7]:
rows = []
for name, buffer in chosen.items():
    table = nightly[name]
    k = pd.Series(oversell(table, buffer), index=table.index)
    for hotel in ["group", *table.index.get_level_values("hotel").unique()]:
        part = table if hotel == "group" else table.xs(hotel, level="hotel", drop_level=False)
        result = contract.night_cost(part, k.loc[part.index])
        rows.append({"hotel": hotel, "method": name, "buffer": buffer, "cost (EUR)": result["cost_eur"],
                     "empty rooms": result["empty_rooms"], "walked guests": result["walked_guests"],
                     "walks per 1,000": result["walks_per_1000"],
                     "ceiling met": result["ceiling_met"] if hotel == "group" else "n/a (no per-hotel threshold)",
                     "hotel-nights": result["nights"]})
results = pd.DataFrame(rows).set_index(["hotel", "method"])

if "flat-rule reference" in chosen:
    flat_cost = results.xs("flat-rule reference", level="method")["cost (EUR)"]
    results["vs flat rule (EUR)"] = results["cost (EUR)"] - flat_cost.reindex(results.index.get_level_values("hotel")).to_numpy()
    results["vs flat rule (%)"] = results["vs flat rule (EUR)"] / flat_cost.reindex(results.index.get_level_values("hotel")).to_numpy()

display(results.style.format({"cost (EUR)": "{:,.0f}", "vs flat rule (EUR)": "{:+,.0f}", "vs flat rule (%)": "{:+.1%}",
                              "walks per 1,000": "{:.2f}"}))

,,buffer,cost (EUR),empty rooms,walked guests,"walks per 1,000",ceiling met,hotel-nights,vs flat rule (EUR),vs flat rule (%)
hotel,method,,,,,,,,,
group,flat-rule reference,16,"2,414,409",5532,169,9.06,True,368,+0,+0.0%
City Hotel,flat-rule reference,16,"1,025,094",3023,167,13.96,n/a (no per-hotel threshold),184,+0,+0.0%
Resort Hotel,flat-rule reference,16,"1,389,315",2509,2,0.30,n/a (no per-hotel threshold),184,+0,+0.0%
group,logistic regression,6,"2,159,327",5098,173,9.28,True,368,"-255,082",-10.6%
City Hotel,logistic regression,6,"1,124,996",3350,117,9.78,n/a (no per-hotel threshold),184,"+99,902",+9.7%
Resort Hotel,logistic regression,6,"1,034,331",1748,56,8.37,n/a (no per-hotel threshold),184,"-354,984",-25.6%
group,random forest,6,"2,036,366",4772,180,9.65,True,368,"-378,044",-15.7%
City Hotel,random forest,6,"1,049,338",3123,116,9.70,n/a (no per-hotel threshold),184,"+24,244",+2.4%
Resort Hotel,random forest,6,"987,028",1649,64,9.57,n/a (no per-hotel threshold),184,"-402,288",-29.0%


## Step 7: how sensitive is each method to its buffer?

Cost and walks per 1,000 for buffers around each method's chosen value, on the group. A method whose cost changes sharply from one room to the next, or that only just meets the ceiling, is fragile. Its buffer may not carry over to the test period.

In [8]:
rows = []
for name, buffer in chosen.items():
    for b in range(max(0, buffer - 3), buffer + 4):
        r = sweep[(sweep["method"] == name) & (sweep["buffer"] == b)].iloc[0]
        rows.append({"method": name, "buffer": b, "chosen": "<-" if b == buffer else "",
                     "cost (EUR)": r["cost_eur"], "walks per 1,000": r["walks_per_1000"], "ceiling met": r["ceiling_met"]})
display(pd.DataFrame(rows).set_index(["method", "buffer"]).style.format({"cost (EUR)": "{:,.0f}", "walks per 1,000": "{:.2f}"}))

## Save the val results

Writes to `data/models/`, beside the training results. `val_chosen_buffers.json` records each method's buffer **as chosen on val**. If the team freezes a method, its buffer is taken from this file, unchanged, for the single test run.

In [9]:
pd.DataFrame(probabilities, index=val.index).to_csv(model_dir / "val_probabilities.csv", index_label="row")
comparison.to_csv(model_dir / "val_scores.csv", index_label="method")
nightly_scores.to_csv(model_dir / "val_nightly_scores.csv")
sweep.to_csv(model_dir / "val_buffer_sweep.csv", index=False)
results.to_csv(model_dir / "val_results.csv")
(model_dir / "val_chosen_buffers.json").write_text(json.dumps({
    "rule": "k = max(0, floor(expected cancellations) - buffer), same buffer every night and both hotels",
    "selection": "lowest val cost among buffers 0-60 meeting 10 walks per 1,000 arrivals (group-wide)",
    "flat_rule_hotel_rates_from_train": {h: float(r) for h, r in hotel_rate.items()},
    "chosen_buffers": chosen,
    "status": "chosen on val (choose period); not frozen; test not run",
}, indent=2) + "\n", encoding="utf-8")

for name in ["val_probabilities.csv", "val_scores.csv", "val_nightly_scores.csv", "val_buffer_sweep.csv",
             "val_results.csv", "val_chosen_buffers.json"]:
    print(f"Saved data/models/{name}")

Saved data/models/val_probabilities.csv
Saved data/models/val_scores.csv
Saved data/models/val_nightly_scores.csv
Saved data/models/val_buffer_sweep.csv
Saved data/models/val_results.csv
Saved data/models/val_chosen_buffers.json


## Summary

- **All four models and the flat rule ran unchanged on val.** No refitting, and test was never opened.
- **Probability scores, calibration and nightly error** show how much of each model's training fit carried over to unseen nights.
- **Every method uses the same oversell rule**, with its buffer chosen on val under the group-wide walk ceiling, and is compared with the flat rule on the same hotel-nights, in euros, empty rooms and walks, for the group and each hotel.

**Before freezing a method:**

1. **Decide and record the choice** in `docs/DECISIONS.md`: the method, its buffer and the reason. The PRD prefers the simpler method when the difference is small.
2. **Remember what the val figures are**: selection figures, not savings. The buffer was chosen on these nights.
3. **Carry the caveats**: four provisional inputs with unverified timing, agent `236` (notebook 10, Step 3), and the full-book caveat.
4. **Then run test once** with the frozen method and buffer, and report euros, empty rooms and walks per 1,000 side by side (project rules 2, 5 and 6).